In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time

import richio
from richio import shockfinder as sf
import dev

## Load snapshot

In [2]:
snap_path = "/home/hey4/rich_tde/data/external/R0.47M0.5BH10000beta1S60Compton/snap_full_361.h5"
snap = richio.load(snap_path)
snap.time.in_units("day")

unyt_array([5.7621411], 'day')

## k-NN graph

<!-- The neighbour graph is only ever queried for the **condition 1 & 2 candidates**
(`∇·v < 0` and `∇T·∇P > 0`) — `_condition3_kernel` runs on them and the surface
ray tracer only walks within the shock zone (a subset). So we compute that
candidate set first and build k-NN rows for just those cells (≈¼ of the box on
TDE data), cutting query time and graph memory ~4×.

This stores a *k*-nearest-neighbour adjacency in CSR format — a clustering-robust,
memory-light drop-in for the global Voronoi build (`sf.build_voronoi`). The
analytical next-cell kernel recovers the Voronoi face per query, so `k`
neighbours (default 48) reproduce the exact-Voronoi shock catalogue while scaling
to the ~80M-cell hi-res runs the full tessellation cannot fit. Built on the full
snapshot so cell indices stay aligned with the snapshot fields. -->

In [3]:
# Candidates = conditions 1 & 2 (the only cells whose neighbours are ever used)
cand = sf.shock_candidates(snap)
print(
    f"Candidates: {len(cand.candidates):,} / {len(snap):,}  ({len(cand.candidates) / len(snap) * 100:.1f}%)"
)

Candidates: 4,118,154 / 17,447,778  (23.6%)


In [4]:
t0 = time.perf_counter()
vor = sf.build_knn(snap, cells=cand.candidates, k=48)
print(f"Built in {time.perf_counter() - t0:.1f} s")

Building k-NN graph (k=48) for 4,118,154 / 17,447,778 cells ...


Done.  48 neighbours/cell.


Built in 49.5 s


## Shock zone

Three criteria from Schaal+15:
1. Converging flow: ∇·v < 0
2. Pressure gradient aligned with temperature gradient: ∇T·∇P > 0
3. Measurable T and P jump across the Voronoi face: Δlog T ≥ 0.11, Δlog P ≥ 0.27

In [5]:
t0 = time.perf_counter()
shock_zone = sf.find_shock_zone(snap, vor)
print(f"{shock_zone.sum():,} shock-zone cells  ({time.perf_counter() - t0:.2f} s)")

432,671 shock-zone cells  (23.35 s)


## Shock surface

For each shock-zone cell, rays are traced along ±ds through the Voronoi graph
until exiting the zone. Mach numbers follow from the Rankine-Hugoniot T, P, ρ jumps.

In [6]:
t0 = time.perf_counter()
result = sf.find_shock_surface(snap, vor, shock_zone)
print(
    f"{result.surface_mask.sum():,} surface cells  ({time.perf_counter() - t0:.2f} s)"
)
print(
    f"Median Mach  T={np.median(result.mach_T):.2f}  P={np.median(result.mach_P):.2f}  rho={np.median(result.mach_rho):.2f}"
)

234,832 surface cells  (14.50 s)
Median Mach  T=2.41  P=1.68  rho=1.13


## Save

Persist the shock-finder output so the analysis notebook
(`0.2-shock-finder-tde-analysis`) can reload it without re-running the finder.
Boolean masks are aligned with the snapshot cell order. The Mach arrays and the
per-surface index arrays (`surf_idx`, `pre_idx`, `post_idx`) are one entry per
shock-surface cell and aligned with each other, so `pre_idx`/`post_idx` give the
upstream/downstream cell for each Mach number (`pre_mask`/`post_mask` are just
their deduplication).

In [7]:
results_path = "/home/hey4/rich_tde/data/interim/shockfinder_R0.47M0.5BH10000beta1S60Compton_361.npz"
np.savez_compressed(
    results_path,
    snap_path=snap_path,
    time_day=float(snap.time.in_units("day").v),
    shock_zone=shock_zone,
    surface_mask=result.surface_mask,
    pre_mask=result.pre_mask,
    post_mask=result.post_mask,
    surf_idx=result.surf_idx,
    pre_idx=result.pre_idx,
    post_idx=result.post_idx,
    mach_T=result.mach_T,
    mach_P=result.mach_P,
    mach_rho=result.mach_rho,
)
print("Saved ->", results_path)

/tmp/ipykernel_1837980/479026449.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  time_day=float(snap.time.in_units("day").v),


Saved -> /home/hey4/rich_tde/data/interim/shockfinder_R0.47M0.5BH10000beta1S60Compton_361.npz
